# ОКС: подготовка данных для моделирования

Входы: локальные CSV без ФИО, контактов, номеров историй болезни и дат.
`group_id` сохраняет группировку повторных госпитализаций. Клинические строки
остаются в исключённых из Git каталогах: это псевдонимизация, а не гарантия анонимности.
Исход: `hospital_death=1` — смерть. Выбран фиксированный набор из 15 признаков.
Импутация, масштабирование и балансировка выполняются только внутри обучения в 02.
Сначала выполните все ячейки 01, затем 02 и 03. Полный пересчёт 02 ресурсоёмок.


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, platform, importlib, datetime
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "studies/acs/configs/layout.json").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри клонированного acs-research-projects")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from acs_research.paths import study_paths
P = study_paths(PROJECT_ROOT)
STUDY = P.study
OUT = P.run
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUT / ".matplotlib"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE_SEED = 2026
VERSIONS = {m: importlib.import_module(m).__version__ for m in
            ["numpy", "pandas", "sklearn", "optuna", "shap", "catboost", "xgboost", "nbclient"]}
print(platform.python_version(), "seed", BASE_SEED, VERSIONS)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savejson(obj, name):
    (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
def figsave(name):
    plt.tight_layout()
    plt.savefig(OUT / (name + ".png"), dpi=180, bbox_inches="tight")
    plt.savefig(OUT / (name + ".svg"), bbox_inches="tight")
    plt.show(); plt.close()


In [ ]:
SRC = P.inputs
csv_path = SRC / "curated_clinical.csv"
raw_path = SRC / "source_measurements.csv"
cur = pd.read_csv(csv_path)
cur.insert(3, "Число койко-дней", np.nan)
cur.columns = cur.columns.str.replace(r"\s+", " ", regex=True).str.strip()
raw = pd.read_csv(raw_path)
raw.columns = raw.columns.str.replace(r"\s+", " ", regex=True).str.strip()
valid = pd.to_numeric(raw["Возраст"], errors="coerce").between(18, 110)
raw = raw.loc[valid].reset_index(drop=True)
assert len(raw) == len(cur) == 906
assert np.array_equal(raw["Возраст"].to_numpy(), cur.iloc[:,5].to_numpy())
assert np.array_equal(raw["Пол"].astype(str).str.strip().map({"М":1,"Ж":0}).to_numpy(), cur.iloc[:,4].to_numpy())
assert cur.iloc[:,2].isin([0,1]).all()
y = (1-cur.iloc[:,2]).astype(int)
assert np.array_equal(y.to_numpy(), (raw["Исход госпитализации"] == 2).astype(int).to_numpy())
def numeric(s): return pd.to_numeric(s.astype(str).str.strip().str.replace(",", ".", regex=False), errors="coerce")
# Curated clinical coding is retained; raw values supersede altered/imputed fields.
indices = {"age":5,"male":4,"mi":0,"st_elevation":1,"tag":6,"ms":7,"killip":21,"low_ef":12,"crp":13,"ckd":11,"pre_tlt":22,"af":9,"pci":23,"grace":24,"diabetes":10}
X = pd.DataFrame({k:numeric(cur.iloc[:,i]) for k,i in indices.items()})
restore = {"tag":"Максимальный уровень ТАГ в ммоль/л","ms":"Метаболический синдром","killip":"Класс Killip","low_ef":"ФВ менее, либо равно 45%","crp":"СРБ","grace":"GRACE","diabetes":"Сахарный диабет"}
for k,c in restore.items(): X[k] = numeric(raw[c])
for k in ["ms","low_ef","diabetes"]: X[k] = X[k].where(X[k].isin([0,1]))
X["killip"] = X.killip.where(X.killip.isin([1,2,3,4]))
# Analytics_1/2 dichotomise intervention: PCI 1/2 versus 0/3.
assert X.pci.isin([0,1,2,3]).all()
X["pci"] = X.pci.map({0:0,1:1,2:1,3:0})
assert X.age.between(18,110).all() and (X.tag.dropna()>0).all()
# Grouping is prepared locally before transfer; names are never read here.
groups = raw["group_id"].to_numpy(dtype=int)
data = X.copy();data.insert(0,"row_id",np.arange(len(data)));data.insert(1,"group_id",groups);data["hospital_death"] = y
assert y.sum() == 43
assert not data.row_id.duplicated().any()
data.to_csv(OUT / "01_данные.csv", index=False)
labels={"age":"Возраст, лет","male":"Мужской пол","mi":"Инфаркт миокарда","st_elevation":"ОКС с подъёмом ST","tag":"Максимальный ТАГ, ммоль/л","ms":"Метаболический синдром","killip":"Класс Killip","low_ef":"ФВ ≤45%","crp":"СРБ, мг/л","ckd":"Хроническая болезнь почек","pre_tlt":"Догоспитальная ТЛТ","af":"Фибрилляция предсердий","pci":"Интервенционное лечение","grace":"GRACE, баллы","diabetes":"Сахарный диабет"}
num=["age","tag","crp","grace"]
cat=[c for c in X if c not in num]
dictionary=pd.DataFrame([dict(feature=k,label=labels[k],source_column=restore.get(k,cur.columns[indices[k]]),source="XLSX" if k in restore else "curated CSV",kind="continuous" if k in num else "categorical",missing=int(X[k].isna().sum()),missing_pct=100*X[k].isna().mean(),unit="ммоль/л" if k=="tag" else "мг/л" if k=="crp" else "лет" if k=="age" else "баллы" if k in ["grace","killip"] else "код",availability="До госпитализации" if k=="pre_tlt" else "Время измерения не установлено",decision="Включён в ретроспективную модель") for k in X])
dictionary.to_csv(OUT / "01_словарь.csv",index=False)
audit={"n":len(data),"deaths":int(y.sum()),"survivors":int((y==0).sum()),"prevalence":float(y.mean()),"groups":int(data.group_id.nunique()),"repeated_groups":int(pd.Series(groups).value_counts().gt(1).sum()),"source_nonclinical_rows_removed":12,"curated_survivors_with_raw_codes_other_than_1":int(((y==0)&(raw["Исход госпитализации"]!=1)).sum()),"tag_values_differing_between_versions":int((abs(numeric(cur.iloc[:,6])-X.tag)>1e-8).sum()),"exact_feature_duplicates":int(X.duplicated().sum()),"raw_ef_invalid":int((numeric(raw["ФВ менее, либо равно 45%"]).notna()&~numeric(raw["ФВ менее, либо равно 45%"]).isin([0,1])).sum())}
files=[csv_path, raw_path]
manifest={str(p.relative_to(PROJECT_ROOT)):sha(p) for p in files}
savejson(manifest,"01_источники_SHA256.json")
savejson(dict(audit=audit,features=list(X),continuous=num,categorical=cat,labels=labels,data_sha256=sha(OUT/"01_данные.csv"),source_sha256=manifest,versions=VERSIONS),"01_конфигурация.json")
exclusions=pd.DataFrame([{"field":c,"reason":"Исход/идентификатор/длительность/лечение с неизвестным кодом/повторная информация или вне согласованного набора"} for c in cur.columns if c not in [cur.columns[i] for i in indices.values()]])
exclusions.to_csv(OUT/"01_исключённые_столбцы.csv",index=False)
print(audit); display(dictionary)
display(data.groupby("hospital_death")[num].agg(["count","median","min","max"]))


## Интерпретация аудита
Пропуск ФВ не равен нормальной ФВ; код 2 не определён заголовком бинарного признака и сохранён неизвестным. Один МС записан как время 00:00, а не диагноз, поэтому сохранён неизвестным. Максимальный ТАГ в CSV отличается от XLSX: используются оригинальные значения без глобального ограничения выбросов. МС не пересчитывается: формальное правило диагноза и полный набор его компонентов отсутствуют. В эксперименте удаления ТАГ/МС диабет остаётся, поэтому часть информации о метаболическом состоянии сохранена.

Группировка по ФИО может соединить тёзок и пропустить варианты написания; это ограничение качества идентификатора. Исторические кодировки диагнозов ХБП/ФП/ТЛТ из CSV сохранены согласно ноутбукам; первичная полнота записи диагнозов не подтверждена. Кодирование 13 выживших с иными raw-кодами основано на существующем анализе автора, а не на придуманном новом словаре. Итог не заявляется прогнозом при поступлении.

In [ ]:
dictionary.set_index("label").missing_pct.sort_values().plot.barh(figsize=(9,6),color="#35658c")
plt.xlabel("Пропуски, %"); plt.title("ОКС: исходная неполнота включённых признаков");figsave("01_пропуски")


## Описательная проверка до интерпретации
Отдельно сохраняется неполнота по исходу и распределение зарегистрированных категорий. В исходном XLSX ФВ≤45% представлена положительными записями и пустыми ячейками: это не полная бинарная оценка ФВ. При импутации модой её клиническая часть может стать константой, а наличие записи отражается флагом пропуска. Анамнестические поля, в которых есть только записи «1» и нерасшифрованные пустые ячейки, не включены в согласованный набор; производный «экстремальный риск» не создаётся. Диабет из текущего диагноза сохранён отдельно от недостаточно документированных пустых анамнестических записей.

In [ ]:
missing_by_outcome=[];observed=[]
for feature in FEATURES if "FEATURES" in globals() else list(X):
    for outcome in [0,1]:
        s=X.loc[y==outcome,feature]
        missing_by_outcome.append(dict(feature=feature,label=labels[feature],hospital_death=outcome,n=len(s),observed=int(s.notna().sum()),missing=int(s.isna().sum()),missing_pct=100*s.isna().mean(),median=s.median(),min=s.min(),max=s.max()))
    observed.append(dict(feature=feature,observed_n=int(X[feature].notna().sum()),unique_observed=int(X[feature].nunique()),min=X[feature].min(),max=X[feature].max(),value_counts=json.dumps(X[feature].value_counts(dropna=False).to_dict(),default=str)))
missing_by_outcome=pd.DataFrame(missing_by_outcome);observed=pd.DataFrame(observed)
missing_by_outcome.to_csv(OUT/"01_описание_по_исходу.csv",index=False);observed.to_csv(OUT/"01_распределения_признаков.csv",index=False)
display(missing_by_outcome)
print("Registered EF codes",X.low_ef.value_counts(dropna=False).to_dict())
